# Lecture 24: Mini-Project: Exploratory Data Analysis (EDA) on a Real Dataset

## Learning Objectives

- Load a real dataset and perform initial inspection
- Clean data by handling missing values, duplicates, and type issues
- Conduct univariate analysis with histograms, box plots, and summary statistics
- Perform bivariate analysis with scatter plots and grouped comparisons
- Explore multivariate patterns with pair plots and correlation matrices
- Draw conclusions and communicate findings in a brief report

## Key Topics

- Loading and inspecting a dataset
- Cleaning: missing values, duplicates, type fixes
- Univariate analysis: histograms, box plots, summary stats
- Bivariate analysis: scatter plots, grouped bars, correlation matrix
- Multivariate patterns with pair plots
- Drawing conclusions and writing a brief report

### Loading and Inspecting the Dataset

We will use the Iris dataset from `sklearn.datasets.load_iris()`, a classic dataset for classification and EDA. It contains 150 samples from three species of iris flowers, with four features: sepal length, sepal width, petal length, and petal width. The dataset is clean and well-documented, making it ideal for demonstrating the full EDA pipeline.

The first step in any EDA is loading the data and performing an initial inspection. We check the shape, data types, missing values, and summary statistics. We also look at the first few rows to verify that the data loaded correctly and to get a sense of the values.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

# Load the Iris dataset
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)

# Initial inspection
print('Shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())
print('\nSummary statistics:')
print(df.describe())

### Data Cleaning

Even though the Iris dataset is clean, we will demonstrate the standard cleaning steps. We check for missing values (there are none), duplicate rows, and ensure all columns have the correct data types. The species column is converted to categorical for efficiency.

In a real dataset, cleaning often involves handling null values, fixing incorrect data types, removing duplicates, and dealing with outliers. These steps ensure that downstream analyses and models are built on reliable data.

In [ ]:
# Data cleaning steps
print('Duplicate rows:', df.duplicated().sum())

# Check for any missing values
if df.isna().sum().sum() == 0:
    print('No missing values found.')

# Verify data types
print('\nSpecies categories:', df['species'].cat.categories.tolist())
print('Species codes:', df['species'].cat.codes[:5])

# Ensure numeric columns are float
for col in iris.feature_names:
    df[col] = df[col].astype(float)
print('\nAll numeric columns confirmed as float64')

### Univariate Analysis

Univariate analysis examines each variable in isolation. We use histograms with KDE overlays to visualise distributions, and box plots to detect outliers and compare spread across species. Summary statistics (`.describe()`) give us the numerical values behind the visualisations.

These plots reveal key insights: petal length and petal width are bimodal (separating setosa from the other two species), while sepal length and width show more overlap. Box plots confirm that setosa is clearly separable on petal features but less so on sepal features.

In [ ]:
# Univariate analysis: histograms
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
features = iris.feature_names

for ax, feature in zip(axes.flat, features):
    sns.histplot(df[feature], kde=True, bins=20, ax=ax)
    ax.set_title(f'Distribution of {feature}')

plt.tight_layout()
plt.show()

In [ ]:
# Univariate analysis: box plots grouped by species
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, feature in zip(axes.flat, features):
    sns.boxplot(x='species', y=feature, data=df, ax=ax,
                palette='Set2')
    ax.set_title(f'{feature} by Species')

plt.tight_layout()
plt.show()

### Bivariate Analysis

Bivariate analysis examines relationships between pairs of variables. Scatter plots coloured by species reveal which feature combinations best separate the classes. The correlation matrix quantifies linear relationships, and grouped bar plots show species-level feature means with error bars.

The petal length vs. petal width scatter plot shows three well-separated clusters. The correlation matrix reveals very high correlation between petal length and petal width (0.96), suggesting redundancy. Sepal width is the least correlated with other features.

In [ ]:
# Bivariate analysis: scatter plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.scatterplot(x='sepal length (cm)', y='sepal width (cm)',
                hue='species', data=df, alpha=0.7, ax=axes[0])
axes[0].set_title('Sepal: Length vs Width')

sns.scatterplot(x='petal length (cm)', y='petal width (cm)',
                hue='species', data=df, alpha=0.7, ax=axes[1])
axes[1].set_title('Petal: Length vs Width')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr = df.select_dtypes(include='number').corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, fmt='.2f')
plt.title('Correlation Matrix - Iris Features')
plt.show()

print('Key finding: petal length and petal width are highly correlated (r=0.96)')

### Multivariate Patterns with Pair Plots

Pair plots provide a comprehensive view of all pairwise relationships in one figure. The diagonal shows distributions for each feature, and off-diagonal cells show scatter plots for each pair. Adding `hue='species'` colour-codes the three iris species, making it easy to see which feature combinations separate the classes.

The pair plot confirms that setosa is linearly separable from versicolor and virginica on petal features alone. Versicolor and virginica overlap somewhat on all features, though petal length and width still provide good separation.

In [ ]:
# Pair plot for multivariate exploration
sns.pairplot(df, hue='species', palette='Set2', diag_kind='kde',
             corner=True, height=2.5)
plt.suptitle('Multivariate Pair Plot of Iris Features', y=1.02)
plt.show()

In [ ]:
# Grouped feature means
means = df.groupby('species', observed=True)[features].mean()
print('Feature means by species:')
print(means.round(2))

# Visualise
means.T.plot(kind='bar', figsize=(10, 6), colormap='Set2', edgecolor='black')
plt.title('Average Feature Values by Species')
plt.ylabel('Centimeters')
plt.xticks(rotation=45)
plt.legend(title='Species')
plt.tight_layout()
plt.show()

### Findings Report

Based on our exploratory data analysis of the Iris dataset, we draw the following conclusions. The dataset contains 150 observations across 3 species with no missing values or duplicates. The key findings are:

1. **Setosa is easily separable**: Iris setosa has distinctly smaller petals (mean length 1.46 cm, width 0.25 cm) and larger sepals (mean width 3.42 cm) compared to the other two species. It forms an isolated cluster in petal feature space.
2. **Versicolor vs virginica overlap**: These two species overlap in all feature dimensions, though they differ in central tendency. Petal length (mean: versicolor 4.26, virginica 5.55) provides the best single-feature separation.
3. **Petal features are more discriminative**: Petal length and petal width show large between-species differences and small within-species variance, making them ideal for classification.
4. **High feature correlation**: Petal length and width are strongly correlated (r=0.96), suggesting they carry redundant information.

**Recommendation**: A classifier using petal length and petal width should achieve near-perfect accuracy on this dataset. Sepal features add limited discriminative power but may help distinguish versicolor from virginica.

In [ ]:
# Summary statistics for the report
summary = df.groupby('species', observed=True)[features].agg(['mean', 'std'])
print('=' * 70)
print('EDA Summary Report - Iris Dataset')
print('=' * 70)
print(f'Dataset size: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Species: {df["species"].cat.categories.tolist()}')
print(f'Samples per species:\n{df["species"].value_counts()}')
print('\nFeature statistics by species:')
print(summary.round(2))
print('\n' + '=' * 70)
print('Conclusion: The Iris dataset is clean, well-structured, and')
print('exhibits clear class separation, especially on petal features.')
print('Petal length and width alone are sufficient for accurate species classification.')

In [ ]:
# Save the cleaned dataset for reference
df.to_csv('iris_clean.csv', index=False)
print('Cleaned Iris dataset saved as iris_clean.csv')

print('\nEDA complete! Key findings:')
print('- No missing values or duplicates')
print('- Setosa is easily separable from the other two species')
print('- Petal features are most discriminative')
print('- High correlation between petal length and petal width')

## Data Science Connection

This mini-project demonstrates the standard EDA workflow used in every data science project. The pattern of loading, inspecting, cleaning, and visualising data is universal across domains. The Iris dataset is a benchmark for classification algorithms, and our EDA confirms that petal features alone can achieve near-perfect separation. The skills you practiced here interpreting distributions, identifying outliers, exploring relationships, and communicating findings are the foundation of data-driven decision-making.